In [ ]:
pip install torch
python ejercicio6_eficiente.py texto.txt

In [ ]:
import math
import sys
import time
from dataclasses import dataclass
from typing import List, Tuple, Dict, Any

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

try:
    from torch.utils.checkpoint import checkpoint
except Exception:
    checkpoint = None

In [ ]:
# -----------------------------
# Datos
# -----------------------------
def load_text(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    return text.replace("\r\n", "\n").replace("\r", "\n")

def build_vocab(text: str):
    chars = sorted(list(set(text)))
    stoi = {ch: i for i, ch in enumerate(chars)}
    itos = {i: ch for ch, i in stoi.items()}
    return stoi, itos

def encode(text: str, stoi: dict) -> torch.Tensor:
    return torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class CharWindowDataset(Dataset):
    def __init__(self, data_ids: torch.Tensor, seq_len: int):
        self.data = data_ids
        self.seq_len = seq_len
    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)
    def __getitem__(self, idx: int):
        x = self.data[idx:idx+self.seq_len]
        y = self.data[idx+1:idx+self.seq_len+1]
        return x, y

In [ ]:
# -----------------------------
# Mini-Transformer causal (con checkpointing opcional)
# -----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3*n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, 1, T, T)
        att = att.masked_fill(mask == 0, float("-inf"))
        att = torch.softmax(att, dim=-1)
        att = self.drop(att)

        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.proj(out)
        out = self.drop(out)
        return out

class MLP(nn.Module):
    def __init__(self, n_embd: int, dropout: float = 0.1):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4*n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4*n_embd, n_embd)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))

class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, dropout: float = 0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size: int, seq_len: int, n_embd: int, n_head: int, n_layer: int,
                 dropout: float = 0.1, use_checkpointing: bool = False):
        super().__init__()
        self.seq_len = seq_len
        self.use_checkpointing = use_checkpointing and (checkpoint is not None)

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(seq_len, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.size()
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = self.drop(tok + pos.unsqueeze(0))

        for blk in self.blocks:
            if self.use_checkpointing:
                x = checkpoint(blk, x)
            else:
                x = blk(x)

        x = self.ln_f(x)
        return self.head(x)

In [ ]:
# -----------------------------
# Entrenamiento eficiente
# -----------------------------
@dataclass
class TrainConfig:
    seq_len: int = 128
    batch_size: int = 32          # batch físico
    accum_steps: int = 1          # acumulación de gradientes
    n_embd: int = 256
    n_head: int = 4
    n_layer: int = 4
    dropout: float = 0.1
    lr: float = 3e-4
    epochs: int = 3
    device: str = "cpu"
    use_amp: bool = False
    use_checkpointing: bool = False


def train_one_epoch(model, loader, optimizer, device, use_amp: bool, accum_steps: int):
    model.train()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0

    use_amp = use_amp and (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    optimizer.zero_grad(set_to_none=True)

    for step, (x, y) in enumerate(loader, start=1):
        x = x.to(device); y = y.to(device)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(x)
            B, T, V = logits.shape
            loss = crit(logits.view(B*T, V), y.view(B*T))
            loss = loss / accum_steps  # promedio por acumulación

        scaler.scale(loss).backward()

        if step % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total += loss.item() * accum_steps
        n += 1

    return total / max(1, n)

@torch.no_grad()
def eval_loss(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    total = 0.0
    n = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = model(x)
        B, T, V = logits.shape
        loss = crit(logits.view(B*T, V), y.view(B*T))
        total += loss.item()
        n += 1
    return total / max(1, n)


def run_experiment(text_path: str, cfg: TrainConfig):
    text = load_text(text_path)
    stoi, _ = build_vocab(text)
    ids = encode(text, stoi)

    n = len(ids)
    split = int(0.9*n)
    train_ids = ids[:split]
    val_ids = ids[split:]

    train_ds = CharWindowDataset(train_ids, cfg.seq_len)
    val_ds = CharWindowDataset(val_ids, cfg.seq_len)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, drop_last=True)

    device = torch.device(cfg.device)

    model = MiniTransformerLM(
        vocab_size=len(stoi),
        seq_len=cfg.seq_len,
        n_embd=cfg.n_embd,
        n_head=cfg.n_head,
        n_layer=cfg.n_layer,
        dropout=cfg.dropout,
        use_checkpointing=cfg.use_checkpointing
    ).to(device)

    optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr)

    # Memoria GPU (si aplica)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    t0 = time.time()
    for epoch in range(1, cfg.epochs+1):
        tr = train_one_epoch(model, train_loader, optim, device, cfg.use_amp, cfg.accum_steps)
        vl = eval_loss(model, val_loader, device)
        ppl = math.exp(vl) if vl < 20 else float("inf")
        print(f"Epoch {epoch:02d} | train_loss={tr:.4f} | val_loss={vl:.4f} | val_ppl={ppl:.2f}")
    t1 = time.time()

    peak_mem = None
    if device.type == "cuda":
        peak_mem = torch.cuda.max_memory_allocated() / (1024**2)  # MiB

    return {
        "val_loss": vl,
        "val_ppl": ppl,
        "time_s": round(t1-t0, 2),
        "peak_mem_mib": round(peak_mem, 2) if peak_mem is not None else None,
        "cfg": cfg
    }

In [ ]:
if __name__ == "__main__":
    text_path = "texto.txt"

    # Define 3 corridas comparables:
    # 1) Baseline
    # 2) Accumulation (mismo batch efectivo)
    # 3) + AMP (si hay GPU) y/o + Checkpointing

    experiments = []

    # Baseline: batch efectivo = 32
    experiments.append(TrainConfig(
        seq_len=128, batch_size=32, accum_steps=1,
        n_embd=256, n_head=4, n_layer=4,
        lr=3e-4, epochs=3, device="cpu",
        use_amp=False, use_checkpointing=False
    ))

    # Accumulation: batch físico 16, accum 2 -> batch efectivo 32
    experiments.append(TrainConfig(
        seq_len=128, batch_size=16, accum_steps=2,
        n_embd=256, n_head=4, n_layer=4,
        lr=3e-4, epochs=3, device="cpu",
        use_amp=False, use_checkpointing=False
    ))

    # Checkpointing + Accumulation (reduce memoria; en CPU se verá más lento)
    experiments.append(TrainConfig(
        seq_len=128, batch_size=16, accum_steps=2,
        n_embd=256, n_head=4, n_layer=4,
        lr=3e-4, epochs=3, device="cpu",
        use_amp=False, use_checkpointing=True
    ))

    # Si quieres GPU, cambia device="cuda" y use_amp=True en alguna corrida.

    results = []
    for i, cfg in enumerate(experiments, start=1):
        print("\n" + "="*80)
        print(f"Experimento {i}: batch={cfg.batch_size}, accum={cfg.accum_steps}, "
              f"AMP={cfg.use_amp}, checkpoint={cfg.use_checkpointing}, device={cfg.device}")
        res = run_experiment(text_path, cfg)
        results.append(res)

    print("\n" + "="*80)
    print("RESUMEN:")
    for r in results:
        c = r["cfg"]
        print({
            "device": c.device,
            "batch": c.batch_size,
            "accum": c.accum_steps,
            "amp": c.use_amp,
            "checkpoint": c.use_checkpointing,
            "val_loss": r["val_loss"],
            "val_ppl": r["val_ppl"],
            "time_s": r["time_s"],
            "peak_mem_mib": r["peak_mem_mib"],
        })